# TopicVisExplorer — End-to-End Tutorial

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezf/TopicVisExplorer/blob/main/examples/00_end_to_end.ipynb)

> **Colab users:** run `!pip install git+https://github.com/gonzalezf/TopicVisExplorer.git` in the first cell (no PyPI package yet).

This notebook walks through the full **bring-your-own-corpus** workflow:

1. Load a CSV corpus (25 short documents bundled with the repo)
2. Fit a topic model (Gensim LDA via the bundled adapter)
3. Inspect top terms per topic
4. Compute topic coherence (NPMI, C_v, segregation, coverage)
5. Run an edit operation (merge two topics) from Python
6. Save a static HTML snapshot of the visualization
7. Learn how to launch the **interactive** browser UI
8. *(Optional)* Load a Hugging Face dataset instead of a local CSV

**Prerequisites (from repo root):**
```bash
uv sync --all-extras
# one-time frontend build (for the interactive server only):
cd frontend && npm ci && npm run build && cd ..
```

All cells in this notebook use the **non-blocking** `tve.prepare()` + `tve.save_html()` path.
`tve.show()` starts a long-running server — see Section 7 for how to launch it from a terminal.

## 1. Version check

In [1]:
import topicvisexplorer as tve

print("topicvisexplorer version:", tve.__version__)

topicvisexplorer version: 1.0.0


## 2. Load a CSV corpus

`sample_corpus.csv` ships in the `examples/` folder. It has 25 short synthetic documents
covering science and technology topics — small enough to fit a model in seconds.

For your own data you need a CSV with a header row and a column containing the document text.
Pass `--csv-text-column <your_column>` to `tve demo --texts` or `byo_csv_text_column=` to
`tve.show(texts_file=...)` (see Section 7).

> **Minimum corpus size:** LDA (Gensim and sklearn) needs at least **50–100 documents**
> to produce coherent, interpretable topics. With fewer documents topics will be noisy
> and NPMI scores will be unreliable. BERTopic needs at least ~50 documents for HDBSCAN
> to find meaningful clusters. The 25-row demo here is intentionally tiny for speed;
> use a larger corpus for real analysis.

In [2]:
from pathlib import Path

import pandas as pd

# Resolve path whether notebook is run from repo root or examples/ dir
_candidates = [Path("examples/sample_corpus.csv"), Path("sample_corpus.csv")]
CSV_PATH = next(p for p in _candidates if p.exists())

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} documents from {CSV_PATH}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Loaded 25 documents from sample_corpus.csv
Columns: ['id', 'text']


,id,text
0,1,machine learning models improve when trained o...
1,2,neural networks learn layers of representation...
2,3,topic models discover hidden themes in large t...


## 3. Fit a topic model (Gensim LDA)

We use `GensimLDAAdapter` — the same adapter the CLI uses for `tve demo --texts --model gensim-lda`.
The adapter returns a `TopicModelData` object with the five arrays `tve.prepare()` needs.

Other adapters follow the same pattern: `SklearnLDAAdapter`, `SklearnNMFAdapter`,
`BERTopicAdapter`, `ETMAdapter`, `CTMAdapter` (see `examples/06_bertopic_show.py` and
`docs/extending.md`).

In [3]:
from gensim.corpora import Dictionary
from gensim.models import LdaModel

from topicvisexplorer.models import GensimLDAAdapter

texts = df["text"].tolist()
tokenized = [t.lower().split() for t in texts]

dictionary = Dictionary(tokenized)
# Filter very rare/common tokens (optional but recommended for real corpora)
dictionary.filter_extremes(no_below=1, no_above=1.0)
corpus = [dictionary.doc2bow(doc) for doc in tokenized]

NUM_TOPICS = 3
lda = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=20,
)

adapter = GensimLDAAdapter()
model_data = adapter.extract(lda, corpus=corpus, dictionary=dictionary)

prepared = tve.prepare(
    topic_term_dists=model_data.topic_term_dists,
    doc_topic_dists=model_data.doc_topic_dists,
    doc_lengths=model_data.doc_lengths,
    vocab=model_data.vocab,
    term_frequency=model_data.term_frequency,
)

K, V = model_data.topic_term_dists.shape
N = model_data.doc_topic_dists.shape[0]
print(f"Fitted LDA: K={K} topics, V={V} vocab terms, N={N} documents")
print(f"topic_term_dists shape: {model_data.topic_term_dists.shape}")
print(f"doc_topic_dists  shape: {model_data.doc_topic_dists.shape}")

Fitted LDA: K=3 topics, V=196 vocab terms, N=25 documents
topic_term_dists shape: (3, 196)
doc_topic_dists  shape: (25, 3)


## 4. Inspect topics

`prepared.topic_top_terms(topic_id, n=10)` returns the top-n terms for a topic
(1-based indexing, matching the browser UI).

In [4]:
print("Top 5 terms per topic (lambda=0.6):\n")
for topic_id in prepared.topic_order:
    terms = prepared.topic_top_terms(topic_id, n=5)
    print(f"  Topic {topic_id}: {', '.join(terms)}")

Top 5 terms per topic (lambda=0.6):

  Topic 1: and, for, herbs, vegetables, fresh
  Topic 2: in, to, and, with, leagues
  Topic 3: of, from, and, with, halls


## 5. Topic coherence

`topicvisexplorer.coherence.report()` computes four metrics:

| Metric | Range | Higher = |
| ------ | ----- | -------- |
| **NPMI** | −1 to 1 | More internally coherent keywords |
| **C_v** | 0 to 1 | Same, sliding-window variant |
| **Segregation** | 0 to 1 | More lexically distinct from other topics |
| **Coverage** | 0 to 1 | More documents assigned to this topic |

These are the same values shown in the collapsible **Coherence** panel in the browser UI.

In [5]:
from topicvisexplorer.coherence import report as coherence_report

coh = coherence_report(
    prepared,
    tokenized_texts=tokenized,
    doc_topic_dists=model_data.doc_topic_dists,
)

rows = []
for i, label in enumerate(coh.labels):
    rows.append({
        "Topic": i + 1,
        "Label": label,
        "NPMI": round(coh.npmi[i], 3),
        "C_v": round(coh.c_v[i], 3),
        "Segregation": round(coh.segregation[i], 3),
        "Coverage": round(coh.coverage[i], 3),
    })

coh_df = pd.DataFrame(rows).set_index("Topic")
print(f"Mean NPMI: {coh.mean_npmi:.3f}   Mean C_v: {coh.mean_c_v:.3f}")
coh_df

Mean NPMI: 0.661   Mean C_v: 0.712


,Label,NPMI,C_v,Segregation,Coverage
Topic,,,,,
1,and for herbs,0.701,0.828,0.947,0.32
2,in to and,0.523,0.615,0.918,0.40
3,of from and,0.761,0.694,0.918,0.28


## 6. Edit operations: merge two topics

`topicvisexplorer.operations.merge()` collapses two topics into one.
It takes the current `prepared` state and the underlying `model_data`, and returns
an updated `(PreparedData, TopicModelData)` pair.

All four operations (`split`, `merge`, `add_word`, `remove_word`) are pure Python
and can be driven from a notebook without a running server.

> **Note on `split`:** `split()` requires a `refit` callable that re-fits a child
> model on the sub-corpus. Use `topicvisexplorer.operations.refit_gensim_lda` or
> pass your own function with signature `(sub_texts: list[str], k_new: int) -> TopicModelData`.
> See `docs/edit-ops.md` for a full example.

In [6]:
from topicvisexplorer.operations import merge

print(f"Before merge: {len(prepared.topic_order)} topics")

# Merge topic 1 and topic 2 (1-based IDs, matching the browser UI)
prepared_merged, model_data_merged = merge(
    prepared,
    topic_id_a=1,
    topic_id_b=2,
    model_data=model_data,
)

print(f"After  merge: {len(prepared_merged.topic_order)} topics")
print("Top terms of merged topic 1:")
print(" ", ", ".join(prepared_merged.topic_top_terms(1, n=6)))

Before merge: 3 topics
After  merge: 2 topics
Top terms of merged topic 1:
  in, and, to, for, with, models


## 7. Save a static HTML snapshot

`tve.save_html()` writes a self-contained HTML file you can open in any browser.
It uses the **legacy** LDAvis-style renderer (no server required).
For the full interactive UI (split, merge, coherence panel, export), use the server
path described below.

In [7]:
from pathlib import Path

out_html = Path("end_to_end_output.html")
tve.save_html(prepared, str(out_html))
print(f"Static HTML saved to: {out_html.resolve()}")
print(f"File size: {out_html.stat().st_size / 1024:.1f} KB")
print()
print("Open in your browser:")
print(f"  file://{out_html.resolve()}")


Static HTML saved to: /root/projects/topicvisexplorer-lib/examples/end_to_end_output.html
File size: 153.5 KB

Open in your browser:
  file:///root/projects/topicvisexplorer-lib/examples/end_to_end_output.html


## 8. Launch the interactive server (run from a terminal)

`tve.show()` starts a **blocking** FastAPI server — the call never returns until you
press Ctrl+C. This means it cannot go in a notebook cell that you expect to finish.
Run it from a separate terminal:

```bash
# Option A: point the CLI at your CSV directly (simplest)
tve demo --texts examples/sample_corpus.csv --csv-text-column text --name csv_demo

# Option B: call tve.show() from a Python script
# (see examples/02_byo_csv_show.py)
uv run python examples/02_byo_csv_show.py
```

Then open **http://127.0.0.1:8000/singlecorpus?scenario=csv_demo&hitl=true** in a browser.

What you'll see:
- **Topic-map scatter** (left) — 2-D PCoA of topic-topic distances; hover to reload the bar chart
- **Top-terms bar chart** (right) — click a circle to pin it and unlock split/merge controls
- **Documents table** (below) — with × buttons to exclude documents
- **Coherence panel** (top-right Σ button) — same NPMI/C_v table as Section 5
- **Export** button — downloads a JSON snapshot to your machine (nothing is uploaded)

> **Jupyter tip:** `tve.show_inline()` (embed without a separate server) is planned for v1.1.
> Until then, use the terminal approach above or open the saved `topics.html` in a browser tab.

## 9. Try a different topic model

Swap `gensim-lda` for `sklearn-nmf` (no extra deps) or `bertopic` / `etm` / `ctm` (`[full]` extras).
The adapter pattern is identical — only the import and fit call change.

In [8]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

from topicvisexplorer.models import SklearnNMFAdapter

vectorizer = TfidfVectorizer(max_df=1.0, min_df=1, max_features=500)
X = vectorizer.fit_transform(texts)

nmf = NMF(n_components=3, random_state=42, max_iter=200, init="nndsvda")
nmf.fit(X)

nmf_model_data = SklearnNMFAdapter().extract(nmf, X, vectorizer=vectorizer)
prepared_nmf = tve.prepare(
    topic_term_dists=nmf_model_data.topic_term_dists,
    doc_topic_dists=nmf_model_data.doc_topic_dists,
    doc_lengths=nmf_model_data.doc_lengths,
    vocab=nmf_model_data.vocab,
    term_frequency=nmf_model_data.term_frequency,
)

print("NMF topics (top 5 terms each):")
for topic_id in prepared_nmf.topic_order:
    terms = prepared_nmf.topic_top_terms(topic_id, n=5)
    print(f"  Topic {topic_id}: {', '.join(terms)}")

NMF topics (top 5 terms each):
  Topic 1: and, herbs, fresh, with, in
  Topic 2: to, the, before, today, end
  Topic 3: from, of, collections, curate, art


## 10. (Optional) Load a Hugging Face dataset

Requires the `[hf]` extra: `pip install -e ".[hf]"` or `pip install datasets`.

The pattern is always the same: load a dataset → write to a local JSONL file →
pass to `tve.show(texts_file=...)` or fit manually as above.
See `examples/05_huggingface_demo.py` for the equivalent runnable script.

The cell below is tagged with a `raises-exception` tag so nbmake skips it gracefully
when `datasets` is not installed.

In [9]:
# This cell requires the [hf] extra:  pip install -e ".[hf]"  or  pip install datasets
import json
import tempfile

try:
    from datasets import load_dataset
    _HF_AVAILABLE = True
except ImportError:
    _HF_AVAILABLE = False
    print("datasets not installed — cell skipped.")
    print('To run this cell:  pip install -e "[hf]"  or  pip install datasets')

if _HF_AVAILABLE:
    ds = load_dataset("ag_news", split="train[:200]")
    hf_texts = [str(row["text"]).strip() for row in ds if str(row["text"]).strip()]
    print(f"Loaded {len(hf_texts)} documents from ag_news train split")

    hf_jsonl = Path(tempfile.mktemp(suffix="-ag_news.jsonl", prefix="tve-"))
    with hf_jsonl.open("w", encoding="utf-8") as f:
        for t in hf_texts:
            f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
    print(f"Wrote {hf_jsonl} ({hf_jsonl.stat().st_size / 1024:.1f} KB)")
    print()
    print("To launch the interactive browser UI, run in a terminal:")
    print(f"  tve demo --texts {hf_jsonl} --name ag_news_demo --num-topics 5")


datasets not installed — cell skipped.
To run this cell:  pip install -e "[hf]"  or  pip install datasets


## Next steps

| Goal | Resource |
| ---- | -------- |
| Interactive browser UI with your CSV | `tve demo --texts your.csv --csv-text-column text` |
| Two-corpus Sankey comparison | `examples/03_two_corpora_sankey.py` |
| Try BERTopic / ETM / CTM from Python | `examples/06_bertopic_show.py` + `docs/extending.md` |
| Split / merge / word edits cookbook | `docs/edit-ops.md` |
| Full flag reference | `tve demo --help` / `docs/own_data.md` |
| Custom topic-model adapter | `docs/extending.md` §1 |